# VolGA / edge-fix training on Google Colab (A100)

Trains the horizon-matched-edge VolGA walk-forward on a cleaned market panel using a Colab A100 GPU.

**Workflow (no manual code/data upload each run):**
1. **Code** is pulled with `git clone` from the public repo (cell 2).
2. **Data** (S&P 500 = Yahoo, redistribution-restricted, NOT on git) is read from **your Google Drive** (cell 3): upload `colab_bundle_sp500_clean.zip` to your Drive **once**, then this notebook mounts Drive and unpacks only the data into the cloned repo. Build the bundle locally with:
   ```
   .venv_gpu_encode/Scripts/python.exe scripts/colab/make_colab_bundle.py --market sp500_clean
   ```
3. Set the runtime to **A100 GPU** (Runtime -> Change runtime type -> A100), then run the cells top to bottom.

In [ ]:
# 0. Confirm the GPU (want A100)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 1. Clone the CODE from the public repo (no manual upload). Shallow clone = fast.
REPO_URL = 'https://github.com/ntquy9901/stock_vol_prediction01.git'
import os, shutil
if os.path.isdir('/content/repo'):
    shutil.rmtree('/content/repo')
!git clone --depth 1 {REPO_URL} /content/repo
print('cloned ->', '/content/repo')
!ls /content/repo/baselines/2026-09-05_edge_horizon_matched/code

In [ ]:
# 2. DATA from Google Drive (upload colab_bundle_sp500_clean.zip to Drive once; NOT committed to git).
#    Adjust DRIVE_ZIP if you placed the bundle in a subfolder of MyDrive.
import os, zipfile
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ZIP = '/content/drive/MyDrive/colab_bundle_sp500_clean.zip'
assert os.path.exists(DRIVE_ZIP), f'bundle not found at {DRIVE_ZIP} -- upload it to your Drive first'
# Extract ONLY the enriched data dir into the cloned repo (code already came from git).
with zipfile.ZipFile(DRIVE_ZIP) as z:
    members = [m for m in z.namelist() if m.startswith('data/processed_enriched/')]
    assert members, 'bundle has no data/processed_enriched/ -- rebuild with make_colab_bundle.py --market sp500_clean'
    z.extractall('/content/repo', members)
print('unpacked', len(members), 'data files from Drive')
!ls /content/repo/data/processed_enriched

In [ ]:
# 3. Deps (Colab ships torch+CUDA; only the light stack may be missing)
!pip -q install pandas numpy scikit-learn
import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 4. Config
MARKET   = 'sp500_clean'          # matches the uploaded data dir
HORIZONS = [1, 5, 10, 22]
FOLDS    = 7                       # retrain cadence target
SEEDS    = 5                       # 5-seed ensemble for the final run
DRIVER   = '/content/repo/baselines/2026-09-05_edge_horizon_matched/code/run_edge_hmatched.py'
print('will train', MARKET, HORIZONS, f'folds={FOLDS} seeds={SEEDS}')

In [ ]:
# 5. Train each horizon (streams progress). Results -> /content/repo/results/edge_hmatched/
import subprocess, time
for H in HORIZONS:
    print(f'\n===== {MARKET} h{H} =====', flush=True)
    t0 = time.time()
    p = subprocess.Popen(['python', DRIVER, '--market', MARKET, '--horizon', str(H),
                          '--folds-target', str(FOLDS), '--n-seeds', str(SEEDS)],
                         cwd='/content/repo', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    p.wait()
    print(f'[h{H}] exit={p.returncode}  {(time.time()-t0)/60:.1f} min')

In [ ]:
# 6. Show the result summaries + download the JSONs
import glob, json
for f in sorted(glob.glob('/content/repo/results/edge_hmatched/*.json')):
    d = json.load(open(f))
    m = d['metrics']
    print(f"{f.split('/')[-1]}: " + ', '.join(f"{k}={m[k]['qlike']:.4f}" for k in m))
    print('   edge density fix vs hm:', d.get('edge_density_fix_mean'), d.get('edge_density_hm_mean'))
import shutil
shutil.make_archive('/content/edge_hmatched_results', 'zip', '/content/repo/results/edge_hmatched')
from google.colab import files as _f
_f.download('/content/edge_hmatched_results.zip')

**After download:** unzip `edge_hmatched_results.zip` into `results/edge_hmatched/` in the local repo; the report/paper builders read those JSONs.

To train a different experiment (e.g. market-dispersion feature), point `DRIVER` at `baselines/2026-09-05_market_disp/code/run_market_disp.py`.
To train a different market, rebuild the bundle with `--market <name>`, upload it to Drive, and set `MARKET` to match (the data cell unpacks whatever `data/processed_enriched/<market>` the bundle contains).